In [4]:

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import pooch
from scipy.sparse import csr_matrix
from scipy.io import mmwrite
import re
import anndata
import csv

In [5]:
cl_terms = [
    {
        "label": "kidney interstitial alternatively activated macrophage",
        "iri": "http://purl.obolibrary.org/obo/CL_1000695",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "kidney distal convoluted tubule epithelial cell",
        "iri": "http://purl.obolibrary.org/obo/CL_1000849",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "epithelial cell of proximal tubule",
        "iri": "http://purl.obolibrary.org/obo/CL_0002306",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "T cell",
        "iri": "http://purl.obolibrary.org/obo/CL_0000084",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "kidney loop of Henle thick ascending limb epithelial cell",
        "iri": "http://purl.obolibrary.org/obo/CL_1001106",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "kidney collecting duct intercalated cell",
        "iri": "http://purl.obolibrary.org/obo/CL_1001432",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "kidney interstitial fibroblast",
        "iri": "http://purl.obolibrary.org/obo/CL_1000692",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "kidney connecting tubule epithelial cell",
        "iri": "http://purl.obolibrary.org/obo/CL_1000768",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "kidney collecting duct principal cell",
        "iri": "http://purl.obolibrary.org/obo/CL_1001431",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "endothelial cell",
        "iri": "http://purl.obolibrary.org/obo/CL_0000115",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "B cell",
        "iri": "http://purl.obolibrary.org/obo/CL_0000236",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "mononuclear phagocyte",
        "iri": "http://purl.obolibrary.org/obo/CL_0000113",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "podocyte",
        "iri": "http://purl.obolibrary.org/obo/CL_0000653",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "parietal epithelial cell",
        "iri": "http://purl.obolibrary.org/obo/CL_1000452",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "mast cell",
        "iri": "http://purl.obolibrary.org/obo/CL_0000097",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "dendritic cell",
        "iri": "http://purl.obolibrary.org/obo/CL_0000451",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "plasma cell",
        "iri": "http://purl.obolibrary.org/obo/CL_0000786",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "monocyte",
        "iri": "http://purl.obolibrary.org/obo/CL_0000576",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "kidney loop of Henle epithelial cell",
        "iri": "http://purl.obolibrary.org/obo/CL_1000909",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "renal interstitial pericyte",
        "iri": "http://purl.obolibrary.org/obo/CL_1001318",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "neutrophil",
        "iri": "http://purl.obolibrary.org/obo/CL_0000775",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "papillary tips cell",
        "iri": "http://purl.obolibrary.org/obo/CL_1000597",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    },
    {
        "label": "neural cell",
        "iri": "http://purl.obolibrary.org/obo/CL_0002319",
        "type": "http://purl.obolibrary.org/obo/CL_0000000",
        "source": "https://www.ebi.ac.uk/ols/ontologies/cl"
    }
]


UBERON_terms = [
    {
        "label": "kidney",
        "iri": "http://purl.obolibrary.org/obo/UBERON_0002113",
        "type": "http://purl.obolibrary.org/obo/UBERON_0000061",
        "source": "https://purl.humanatlas.io/vocab/hp"
    },
    {
        "label": "nephron tubule",
        "iri": "http://purl.obolibrary.org/obo/UBERON_0001231",
        "type": "http://purl.obolibrary.org/obo/UBERON_0000061",
        "source": "https://purl.humanatlas.io/vocab/hp"
    },
    {
        "label": "collecting duct of renal tubule",
        "iri": "http://purl.obolibrary.org/obo/UBERON_0001232",
        "type": "http://purl.obolibrary.org/obo/UBERON_0000061",
        "source": "https://purl.humanatlas.io/vocab/hp"
    },
    {
        "label": "endothelium",
        "iri": "http://purl.obolibrary.org/obo/UBERON_0001986",
        "type": "http://purl.obolibrary.org/obo/UBERON_0000061",
        "source": "https://purl.humanatlas.io/vocab/hp"
    },
    {
        "label": "renal glomerulus",
        "iri": "http://purl.obolibrary.org/obo/UBERON_0000074",
        "type": "http://purl.obolibrary.org/obo/UBERON_0000061",
        "source": "https://purl.humanatlas.io/vocab/hp"
    },
    {
        "label": "renal corpuscle",
        "iri": "http://purl.obolibrary.org/obo/UBERON_0001229",
        "type": "http://purl.obolibrary.org/obo/UBERON_0000061",
        "source": "https://purl.humanatlas.io/vocab/hp"
    },
    {
        "label": "kidney capillary",
        "iri": "http://purl.obolibrary.org/obo/UBERON_0003527",
        "type": "http://purl.obolibrary.org/obo/UBERON_0000061",
        "source": "https://purl.humanatlas.io/vocab/hp"
    },
    {
        "label": "renal papilla",
        "iri": "http://purl.obolibrary.org/obo/UBERON_0001228",
        "type": "http://purl.obolibrary.org/obo/UBERON_0000061",
        "source": "https://purl.humanatlas.io/vocab/hp"
    }
]

cell_to_organ_mapping = {
    "proximal tubule": "nephron tubule",
    "distal convoluted tubule": "nephron tubule",
    "loop of Henle": "nephron tubule",
    "connecting tubule": "nephron tubule",
    "collecting duct principal": "collecting duct of renal tubule",
    "collecting duct intercalated": "collecting duct of renal tubule",
    "podocyte": "renal glomerulus",
    "parietal epithelial cell": "renal corpuscle",
    "papillary tips": "renal papilla",
    "endothelial": "endothelium",
    "capillary": "endothelium",
    "default": "kidney"
}


In [ ]:
organ_lookup = {o["label"]: o["iri"] for o in UBERON_terms}
rows = []
for cell in cl_terms:
    label = cell["label"]
    iri = cell["iri"]

    organ_label = next((v for k, v in cell_to_organ_mapping.items() if k in label), cell_to_organ_mapping["default"])

    organ_iri = organ_lookup[organ_label]

    rows.append([
        iri,
        "http://purl.org/ccf/ccf_located_in",
        organ_iri,
        "https://purl.humanatlas.io/collection/hra"
    ])

with open("hra-kpmp-hubmap-cell-to-organ-edges.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["subject", "predicate", "object", "source"])
    writer.writerows(rows)

In [10]:
rows_nodes = []

for i in UBERON_terms:
    rows_nodes.append([
        i["iri"],
        i["label"],
        "http://purl.obolibrary.org/obo/UBERON_0000061",
        "https://purl.humanatlas.io/vocab/hp"
    ])

for i in cl_terms:
    rows_nodes.append([
        i["iri"],
        i["label"],
        "http://purl.obolibrary.org/obo/CL_0000000",
        "https://purl.humanatlas.io/graph/hra-lit"
    ])

with open("hra-kpmp-hubmap-cell-to-organ-nodes.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["iri", "label", "type", "source"])
    writer.writerows(rows_nodes)
